# 第18章：条件波动率、预测时钟与代理损失

本 Notebook 按顺序运行一项手算核验、根拥有的 shared volatility core 案例、真实 French NoDur 历史窗口分析和四张实际图。收益是月度小数，方差是小数月收益平方。模拟真方差与真实市场代理分开展示。

执行命令（课程根目录）：

```bash
.venv/bin/python -m jupyter nbconvert --to notebook --execute --inplace notebooks/ch18_volatility.ipynb --ExecutePreprocessor.kernel_name=quant-course-venv
```

或用 Notebook UI 从首格执行。研究窗口为训练 1990-01 至 2006-12、评价 2007-01 至 2009-12；它是当前冻结快照上的历史归因，不是 PIT、预注册或当时实时可交易记录。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch18_volatility_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(ROOT / "code"))

import ch18_volatility as workflow
from quant_course.volatility import (
    GARCHFit, GARCHParameters, ewma_variance_path, garch_variance_forecast,
    garch_variance_path, realized_variance,
)
print("Project root:", ROOT)

## 1. 先核手算：一段路径不是净收益平方

同一区间三个子区间的对数收益为 1%、−2%、3%。平方和是该网格的 RV；净对数收益平方只记录净位移。EWMA/ARCH/GARCH手算则把同样的三个数解释为逐月收益创新，目标不同。随后用共享核心核验 EWMA、ARCH/GARCH 和逐期多步预测。

In [ ]:
parts = [0.01, -0.02, 0.03]
rv = realized_variance(parts)
net_squared = sum(parts) ** 2
assert abs(rv - 0.0014) < 1e-12
assert abs(net_squared - 0.0004) < 1e-12
idx = pd.period_range("2000-01", periods=3, freq="M")
shocks = pd.Series(parts, index=idx)
params = GARCHParameters(omega=0.00001, alpha=0.1, beta=0.8)
ewma = ewma_variance_path(shocks, decay=0.9, initial_variance=0.0004)
garch = garch_variance_path(shocks, params, initial_variance=0.0004)
arch = garch_variance_path(shocks, GARCHParameters(0.00001, 0.1, 0.0), initial_variance=0.0004)
toy_fit = GARCHFit(params, 0.0, pd.Period("2000-03", freq="M"), 60, 0.0004, 0.0, 1, garch)
multi_step = garch_variance_forecast(toy_fit, steps=3)
assert abs(ewma.iloc[0] - 0.0004) < 1e-12
assert abs(ewma.iloc[1] - 0.00037) < 1e-12
assert abs(ewma.iloc[2] - 0.000373) < 1e-12
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    arch.tolist(), [0.0004, 0.00002, 0.00005]))
assert all(abs(actual - expected) < 1e-12 for actual, expected in zip(
    garch.tolist(), [0.0004, 0.00034, 0.000322]))
print({"RV": rv, "net_return_squared": net_squared,
       "EWMA h1:h3": ewma.tolist(), "ARCH h1:h3": arch.tolist(),
       "GARCH h1:h3": garch.tolist(),
       "GARCH long-run h": params.unconditional_variance,
       "expected one-month h for horizons 1:3": multi_step.tolist()})

**读数**：RV 与净对数收益平方差别来自方向抵消。EWMA/GARCH 返回同一单位的逐期方差路径；将其开平方才得到小数收益标准差。共享核心的输入创新在每期结束后才用于更新后一期。

In [ ]:
summary, simulation = workflow.analyze_constructed()
simulation.to_csv(RUN_DIR / "constructed_simulation.csv")
print(summary)
assert summary["simulation"]["observations"] == 360


## 2. 样本契约与历史估计

该命令刚刚从冻结 manifest 校验 raw ZIP、解压 CSV 和处理 CSV；若任何字节变化，它会停止而不会联网下载。固定 GARCH 参数只拟合 204 个训练月。评价期36个目标月用上月已实现冲击更新递推；当月平方收益代理则在目标月结束后才用于损失计算。

In [ ]:
if RUN_FRENCH:
    actual = pd.read_csv(ROOT / "outputs/ch18/ch18_actual_forecasts.csv", parse_dates=False)
    actual.head(), actual.tail(), len(actual)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 3. 如何读真实输出，及其边界

QLIKE 表中最低代理损失不是交易策略冠军。真实序列没有潜在逐月条件方差标签；目标月平方创新是嘈杂代理，而且训练均值若偏离评价期均值会引入条件偏差。测试期只有36个月且是危机历史切片，不做胜者显著性结论。图中的目标月 forecast 仅依赖此前回报；但源数据的历史发布日期/修订版本没有归档，所以它仍不是 PIT 或可执行的当时实盘 OOS 记录。没有仓位或成交账本，成本后策略收益不适用，而不是成本为零。

模拟图另有完整 `true_variance` 序列，可核对已知 GARCH 数据生成过程；有限 burn-in 只减弱初始化影响，不保证精确平稳抽样。模拟结果是代码验证，不是市场证据。

In [ ]:
simulation_summary = summary["simulation"]
simulation_summary